# GNN 以外の軽い実験（提出ファイル最大20本）

**目的**: 実装済みの改善候補と ModernBERT で最大20本の提出ファイルを出して実験する。ModernBERT は処理が重いため **fold ごとにキャッシュ** し、再実行で未完了 fold から再開できる。

**参照**: `docs/16_NEXT_HEAVY_IMPROVEMENTS.md`, `docs/11_WHAT_NEXT_AFTER_HIGH_POTENTIAL.md`

| # | 実験内容 | 提出ファイル |
|---|----------|-------------|
| 1 | 類似映画を何本レビューしたか | submission_similar_movies_reviewed.csv |
| 2 | 類似度 TE（02） | submission_improvement_02_similarity_te.csv |
| 3 | scale_pos_weight（05） | submission_improvement_05_scale_pos_weight.csv |
| 4 | 特徴選択（07） | submission_improvement_07_feature_selection.csv |
| 5 | 弱点 1 列（10） | submission_improvement_10_extra_col.csv |
| 6 | 擬似ラベル閾値見直し（01） | submission_improvement_01_pseudo_label.csv |
| 7 | TF-IDF+SVD（08） | submission_improvement_08_tfidf_svd.csv |
| 8 | BPR64 スタッキング（03） | submission_improvement_03_stacking.csv |
| 9 | 重み付きブレンド（3本均等） | submission_blend_weighted_3.csv |
| 10 | BPR64 シード平均 | submission_bpr64_seed_avg.csv |
| 11 | ModernBERT（Stratified 2-fold） | submission_modernbert.csv |
| 12 | ModernBERT（時系列 2-fold） | submission_modernbert_time.csv |
| 13 | DeBERTa v3 base | submission_deberta_v3_base.csv |
| 14 | ModernBERT（FILL_MAP なし） | submission_modernbert_nofill.csv |
| 15 | 最高精度 + ModernBERT（0.5/0.5） | submission_blend_best_bert.csv |
| 16 | 最高精度 + ModernBERT（0.7/0.3） | submission_blend_best_bert_07.csv |
| 17 | 最高精度 + ModernBERT（0.3/0.7） | submission_blend_best_bert_03.csv |
| 18 | 最高精度 + DeBERTa base（0.5/0.5） | submission_blend_best_deberta.csv |
| 19 | 4本均等（bpr64, als64, bpr128, ModernBERT） | submission_blend_weighted_4.csv |
| 20 | 最高精度 + ModernBERT（0.6/0.4） | submission_blend_best_bert_06.csv |

**土台の整理**（最高精度 0.76591 = bpr64_count1 + BPR128 のブレンド）:
- **#1–8, #10**: いずれも **最高精度と同じ土台**。左側＝bpr64_count1（#1 は bpr64_count1+類似映画 1 列、#2–8/#10 は bpr64_count1+各改善）の 1 本を出したあと、**BPR128 と均等ブレンド**して同じファイル名で保存。提出は「〇〇 + BPR128」の 2 本ブレンド＝0.76591 と同じ形式。
- **#9**: 3本均等ブレンド（bpr64_only, als64, bpr128）。最高精度提出そのものではない。
- **#11–14**: BERT/ModernBERT 単体（ctx_bert）。
- **#15–18, #20**: **最高精度提出 CSV（0.76591）** と BERT の予測を加重平均したブレンド。
- 既存ファイルがある実験はスキップ。ModernBERT は `outputs/bert_cache/` で再開可能。

## セットアップ

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from lib.improvement_candidates import (
    get_setup,
    get_bpr_base,
    run_01_pseudo_label,
    run_02_similarity_te,
    run_03_stacking,
    run_05_scale_pos_weight,
    run_07_feature_selection,
    run_08_tfidf_svd,
    run_10_extra_col,
    run_similar_movies_reviewed,
)
from lib.submission import save_submission, verify_submission, blend_two_submissions
from lib.two_hop import add_2hop_features, TWO_HOP_REVIEW_COUNT, run_experiment

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)
print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")

# BPR64 単体（参照用）。#1 は ctx_bpr_count1 + use_existing_features を使用
train_df, test_df, feats = get_bpr_base(ctx, factors=64)
for col in feats:
    if not pd.api.types.is_numeric_dtype(train_df[col]) and not pd.api.types.is_categorical_dtype(train_df[col]):
        train_df[col] = train_df[col].astype("category")
        test_df[col] = test_df[col].astype("category")
ctx_bpr = ctx._replace(X=train_df[feats], X_test=test_df[feats], features=feats)

# 最高精度の片方・bpr64_count1 ベース（55 + BPR64 + movie_review_count）。#1–8, #10 で使用
# run_02 等は ctx.train / ctx.test から ctx.features を参照するため、train/test も差し替える
train_c1, test_c1, feats_c1 = get_bpr_base(ctx, factors=64)
add_2hop_features(train_c1, test_c1, columns=[TWO_HOP_REVIEW_COUNT])
feats_c1 = feats_c1 + [TWO_HOP_REVIEW_COUNT]
for col in feats_c1:
    if not pd.api.types.is_numeric_dtype(train_c1[col]) and not pd.api.types.is_categorical_dtype(train_c1[col]):
        train_c1[col] = train_c1[col].astype("category")
        test_c1[col] = test_c1[col].astype("category")
ctx_bpr_count1 = ctx._replace(train=train_c1, test=test_c1, X=train_c1[feats_c1], X_test=test_c1[feats_c1], features=feats_c1)
print(f"BPR64 特徴数: {len(ctx_bpr.features)}, bpr64_count1 特徴数: {len(ctx_bpr_count1.features)}（#1–8,#10 の土台）")

# 最高精度 0.76591 = bpr64_count1 と BPR128 のブレンド。BPR128 が無ければ生成する。
path_bpr128 = ctx.submissions_dir / "submission_2hop_bpr128_only.csv"
if not path_bpr128.exists():
    r_bpr128 = run_experiment(ctx, "bpr128_only", bpr_factors=128)
    print(f"  [BPR128] → {path_bpr128.name}  ({'OK' if r_bpr128.get('ok') else r_bpr128.get('message')})")
else:
    print(f"  [BPR128] 既存: {path_bpr128.name}")

def blend_with_bpr128(path_improvement: Path):
    """改善 1 本を BPR128 と均等ブレンドし、同じ path に上書き（最高精度と同じ形式）。"""
    return blend_two_submissions(
        path_improvement, path_bpr128, path_improvement,
        weight_a=0.5, test_ids=ctx.test["ID"].values,
    )

print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

BPR64 特徴数: 183, bpr64_count1 特徴数: 184（#2–8,#10 の土台）
提出先: outputs/submissions


## 1. 類似映画を何本レビューしたか

In [2]:
path = ctx.submissions_dir / "submission_similar_movies_reviewed.csv"
if not path.exists():
    # 最高精度の片方 bpr64_count1 に類似映画 1 列を足す（use_existing_features=True）
    r = run_similar_movies_reviewed(ctx_bpr_count1, top_k=20, use_existing_features=True)
    if r.get("ok"):
        blend_with_bpr128(path)
    print(f"  [類似映画] → {path.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [類似映画] 既存ファイルを使用: {path.name}")

  0%|          | 0/100 [00:00<?, ?it/s]

  [類似映画] → submission_similar_movies_reviewed.csv  (OK)


## 2. 類似度 TE（02）

In [3]:
path

PosixPath('outputs/submissions/submission_similar_movies_reviewed.csv')

In [ ]:
path = ctx.submissions_dir / "submission_improvement_02_similarity_te.csv"
if not path.exists():
    r = run_02_similarity_te(ctx_bpr_count1, k=5)
    if r.get("ok"):
        blend_with_bpr128(path)
    print(f"  [02 類似度TE] → {path.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [02 類似度TE] 既存ファイルを使用: {path.name}")

## 3. scale_pos_weight（05）・4. 特徴選択（07）・5. 弱点 1 列（10）

In [ ]:
path_05 = ctx.submissions_dir / "submission_improvement_05_scale_pos_weight.csv"
if not path_05.exists():
    r = run_05_scale_pos_weight(ctx_bpr_count1)
    if r.get("ok"):
        blend_with_bpr128(path_05)
    print(f"  [05 scale_pos_weight] → {path_05.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [05 scale_pos_weight] 既存ファイルを使用: {path_05.name}")

path_07 = ctx.submissions_dir / "submission_improvement_07_feature_selection.csv"
if not path_07.exists():
    r = run_07_feature_selection(ctx_bpr_count1)
    if r.get("ok"):
        blend_with_bpr128(path_07)
    print(f"  [07 特徴選択] → {path_07.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [07 特徴選択] 既存ファイルを使用: {path_07.name}")

path_10 = ctx.submissions_dir / "submission_improvement_10_extra_col.csv"
if not path_10.exists():
    r = run_10_extra_col(ctx_bpr_count1)
    if r.get("ok"):
        blend_with_bpr128(path_10)
    print(f"  [10 弱点1列] → {path_10.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [10 弱点1列] 既存ファイルを使用: {path_10.name}")

  [05 scale_pos_weight] → submission_improvement_05_scale_pos_weight.csv  (OK)
  [07 特徴選択] → submission_improvement_07_feature_selection.csv  (OK)
  [10 弱点1列] → submission_improvement_10_extra_col.csv  (OK)


## 6. 擬似ラベル閾値見直し（01）・7. TF-IDF+SVD（08）

In [ ]:
path_01 = ctx.submissions_dir / "submission_improvement_01_pseudo_label.csv"
if not path_01.exists():
    r = run_01_pseudo_label(ctx_bpr_count1, high_thresh=0.98, low_thresh=0.02, max_pseudo=1500, pseudo_weight=0.3)
    if r.get("ok"):
        blend_with_bpr128(path_01)
    print(f"  [01 擬似ラベル] → {path_01.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [01 擬似ラベル] 既存ファイルを使用: {path_01.name}")

path_08 = ctx.submissions_dir / "submission_improvement_08_tfidf_svd.csv"
if not path_08.exists():
    r = run_08_tfidf_svd(ctx_bpr_count1)
    if r.get("ok"):
        blend_with_bpr128(path_08)
    print(f"  [08 TF-IDF+SVD] → {path_08.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
else:
    print(f"  [08 TF-IDF+SVD] 既存ファイルを使用: {path_08.name}")

  [01 擬似ラベル] → submission_improvement_01_pseudo_label.csv  (OK)
  [08 TF-IDF+SVD] → submission_improvement_08_tfidf_svd.csv  (OK)


## 8. BPR64 スタッキング（03）

In [ ]:
path_03 = ctx.submissions_dir / "submission_improvement_03_stacking.csv"
if not path_03.exists():
    r = run_03_stacking(ctx_bpr_count1, seed=None)
    if r.get("path") and r.get("ok"):
        blend_with_bpr128(path_03)
    if r.get("path"):
        print(f"  [03 スタッキング] → {path_03.name}  ({'OK' if r.get('ok') else r.get('message')}) (BPR128 とブレンド済み)")
    else:
        print(f"  [03 スタッキング] スキップ: {r.get('message')}")
else:
    print(f"  [03 スタッキング] 既存ファイルを使用: {path_03.name}")

CatBoostError: features data: pandas.DataFrame column 'rotten_tomatoes_link' has dtype 'category' but is not in  cat_features list

## 9. 重み付きブレンド（3本均等）

In [13]:
out_blend = ctx.submissions_dir / "submission_blend_weighted_3.csv"
paths_3 = [
    ctx.submissions_dir / "submission_2hop_bpr64_only.csv",
    ctx.submissions_dir / "submission_atmacup_implicit_als64.csv",
    ctx.submissions_dir / "submission_2hop_bpr128_only.csv",
]
if out_blend.exists():
    print(f"  [重み付きブレンド 3本] 既存ファイルを使用: {out_blend.name}")
elif all(p.exists() for p in paths_3):
    dfs = []
    for p in paths_3:
        df = pd.read_csv(p)[["ID", "target"]].rename(columns={"target": p.stem})
        dfs.append(df)
    m = dfs[0]
    for d in dfs[1:]:
        m = m.merge(d, on="ID")
    target_cols = [c for c in m.columns if c != "ID"]
    m["target"] = m[target_cols].mean(axis=1)
    m = m[["ID", "target"]].set_index("ID").reindex(ctx.test["ID"]).reset_index()
    save_submission(ctx.test, m["target"].values, out_blend, sanitize=True)
    print(f"  [重み付きブレンド 3本] → {out_blend.name}  (OK)")
else:
    missing = [p.name for p in paths_3 if not p.exists()]
    print(f"  [重み付きブレンド 3本] スキップ: ファイル不足 {missing}")

  [重み付きブレンド 3本] スキップ: ファイル不足 ['submission_2hop_bpr64_only.csv', 'submission_atmacup_implicit_als64.csv', 'submission_2hop_bpr128_only.csv']


## 10. BPR64 シード平均

In [14]:
import lightgbm as lgb

path_seed_avg = ctx.submissions_dir / "submission_bpr64_seed_avg.csv"
if not path_seed_avg.exists():
    preds = []
    for s in [42, 43, 44]:
        params = {**ctx_bpr_count1.lgb_params, "random_state": s}
        model = lgb.LGBMClassifier(**params)
        model.fit(ctx_bpr_count1.X, ctx_bpr_count1.y)
        preds.append(model.predict_proba(ctx_bpr_count1.X_test)[:, 1])
    pred_avg = np.mean(preds, axis=0)
    save_submission(ctx.test, pred_avg, path_seed_avg, sanitize=True)
    blend_with_bpr128(path_seed_avg)
    print(f"  [BPR64 シード平均] → {path_seed_avg.name}  (OK) (BPR128 とブレンド済み)")
else:
    print(f"  [BPR64 シード平均] 既存ファイルを使用: {path_seed_avg.name}")

  [BPR64 シード平均] → submission_bpr64_seed_avg.csv  (OK)


## 11–14. ModernBERT / BERT 単体（重い・fold キャッシュで再開可）

各実験は提出 CSV が既にあればスキップ。なければ `outputs/bert_cache/` に fold ごとに予測を保存するので、中断しても再実行で未完了 fold から再開できる。ログで `[BERT] xxx fold 1/2 学習中...` / `[BERT cache] xxx fold 1/2 を読み込み` を確認できる。

In [16]:
# BERT 用 ctx（embedding を読まないので軽い）。キャッシュで再開可能
from lib.improvement_candidates import run_bert_deberta_submission

ctx_bert = get_setup(seed=42, output_dir="outputs", use_best_pipeline=False)
cache_dir = ctx_bert.submissions_dir.parent / "bert_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
print("BERT 提出先:", ctx_bert.submissions_dir, "キャッシュ:", cache_dir)

# 11. ModernBERT（Stratified 2-fold）
path_11 = ctx_bert.submissions_dir / "submission_modernbert.csv"
if not path_11.exists():
    r = run_bert_deberta_submission(ctx_bert, out_name="submission_modernbert.csv", cache_dir=cache_dir, cache_name="modernbert")
    print("  [11]", "→ submission_modernbert.csv" if r.get("ok") else r.get("message"))
else:
    print("  [11] 既存: submission_modernbert.csv")

# 12. ModernBERT（時系列 2-fold）
path_12 = ctx_bert.submissions_dir / "submission_modernbert_time.csv"
if not path_12.exists():
    r = run_bert_deberta_submission(ctx_bert, cv_strategy="time", out_name="submission_modernbert_time.csv", cache_dir=cache_dir, cache_name="modernbert_time")
    print("  [12]", "→ submission_modernbert_time.csv" if r.get("ok") else r.get("message"))
else:
    print("  [12] 既存: submission_modernbert_time.csv")

# 13. DeBERTa v3 base
path_13 = ctx_bert.submissions_dir / "submission_deberta_v3_base.csv"
if not path_13.exists():
    r = run_bert_deberta_submission(ctx_bert, model_name="microsoft/deberta-v3-base", out_name="submission_deberta_v3_base.csv", cache_dir=cache_dir, cache_name="deberta_v3_base")
    print("  [13]", "→ submission_deberta_v3_base.csv" if r.get("ok") else r.get("message"))
else:
    print("  [13] 既存: submission_deberta_v3_base.csv")

# 14. ModernBERT（FILL_MAP なし）
path_14 = ctx_bert.submissions_dir / "submission_modernbert_nofill.csv"
if not path_14.exists():
    r = run_bert_deberta_submission(ctx_bert, use_fill_map=False, out_name="submission_modernbert_nofill.csv", cache_dir=cache_dir, cache_name="modernbert_nofill")
    print("  [14]", "→ submission_modernbert_nofill.csv" if r.get("ok") else r.get("message"))
else:
    print("  [14] 既存: submission_modernbert_nofill.csv")

BERT 提出先: outputs/submissions キャッシュ: outputs/bert_cache


TypeError: Cannot setitem on a Categorical with a new category ([NO_TITLE]), set the categories first

## 15–20. 最高精度 + BERT ブレンド（複数パターン）

- **#15, #16, #17, #18, #20**: **最高精度（Public 0.76591）** の提出 `submission_blend_bpr64_count1_bpr128.csv`（bpr64_count1 + BPR128 の均等ブレンド）の**予測**と、ModernBERT/DeBERTa の**予測**を加重平均した提出。  
  - 式: `提出確率 = weight_best × 最高精度CSVのtarget + (1 - weight_best) × BERTのtarget`  
  - 最高精度から「上がりうる」かは LB で確認。ブレンド重みは #16(0.7/0.3), #17(0.3/0.7), #20(0.6/0.4) で複数パターン出して比較可能。
- **#19**: 最高精度提出は使わず、4本（bpr64_only, als64, bpr128, ModernBERT）の均等ブレンド。比較用。
- 既存ファイルがあればスキップ。

In [ ]:
# 15–20: ブレンド提出（最高精度 + BERT など）
from lib.improvement_candidates import run_bert_blend_with_best

best_name = "submission_blend_bpr64_count1_bpr128.csv"
sub_dir = ctx.submissions_dir

# 15. 最高精度 + ModernBERT（0.5 / 0.5）
out_15 = sub_dir / "submission_blend_best_bert.csv"
if not out_15.exists():
    r = run_bert_blend_with_best(ctx, best_name=best_name, bert_name="submission_modernbert.csv", out_name="submission_blend_best_bert.csv", weight_best=0.5)
    print("  [15]", "OK" if r.get("ok") else r.get("message"))
else:
    print("  [15] 既存: submission_blend_best_bert.csv")

# 16. 最高精度 0.7 + ModernBERT 0.3
out_16 = sub_dir / "submission_blend_best_bert_07.csv"
if not out_16.exists():
    r = run_bert_blend_with_best(ctx, best_name=best_name, bert_name="submission_modernbert.csv", out_name="submission_blend_best_bert_07.csv", weight_best=0.7)
    print("  [16]", "OK" if r.get("ok") else r.get("message"))
else:
    print("  [16] 既存: submission_blend_best_bert_07.csv")

# 17. 最高精度 0.3 + ModernBERT 0.7
out_17 = sub_dir / "submission_blend_best_bert_03.csv"
if not out_17.exists():
    r = run_bert_blend_with_best(ctx, best_name=best_name, bert_name="submission_modernbert.csv", out_name="submission_blend_best_bert_03.csv", weight_best=0.3)
    print("  [17]", "OK" if r.get("ok") else r.get("message"))
else:
    print("  [17] 既存: submission_blend_best_bert_03.csv")

# 18. 最高精度 + DeBERTa base（0.5 / 0.5）
out_18 = sub_dir / "submission_blend_best_deberta.csv"
if not out_18.exists():
    r = run_bert_blend_with_best(ctx, best_name=best_name, bert_name="submission_deberta_v3_base.csv", out_name="submission_blend_best_deberta.csv", weight_best=0.5)
    print("  [18]", "OK" if r.get("ok") else r.get("message"))
else:
    print("  [18] 既存: submission_blend_best_deberta.csv")

# 19. 4本均等（bpr64_only, als64, bpr128, ModernBERT）
out_19 = sub_dir / "submission_blend_weighted_4.csv"
paths_4 = [sub_dir / "submission_2hop_bpr64_only.csv", sub_dir / "submission_atmacup_implicit_als64.csv", sub_dir / "submission_2hop_bpr128_only.csv", sub_dir / "submission_modernbert.csv"]
if not out_19.exists() and all(p.exists() for p in paths_4):
    dfs = [pd.read_csv(p)[["ID", "target"]].rename(columns={"target": p.stem}) for p in paths_4]
    m = dfs[0]
    for d in dfs[1:]:
        m = m.merge(d, on="ID")
    m["target"] = m[[c for c in m.columns if c != "ID"]].mean(axis=1)
    m = m[["ID", "target"]].set_index("ID").reindex(ctx.test["ID"]).reset_index()
    save_submission(ctx.test, m["target"].values, out_19, sanitize=True)
    print("  [19] → submission_blend_weighted_4.csv")
else:
    print("  [19]", "既存" if out_19.exists() else f"スキップ（不足: {[p.name for p in paths_4 if not p.exists()]})")

# 20. 最高精度 0.6 + ModernBERT 0.4
out_20 = sub_dir / "submission_blend_best_bert_06.csv"
if not out_20.exists():
    r = run_bert_blend_with_best(ctx, best_name=best_name, bert_name="submission_modernbert.csv", out_name="submission_blend_best_bert_06.csv", weight_best=0.6)
    print("  [20]", "OK" if r.get("ok") else r.get("message"))
else:
    print("  [20] 既存: submission_blend_best_bert_06.csv")

## 提出ファイル一覧

In [ ]:
expected = [
    "submission_similar_movies_reviewed.csv",
    "submission_improvement_02_similarity_te.csv",
    "submission_improvement_05_scale_pos_weight.csv",
    "submission_improvement_07_feature_selection.csv",
    "submission_improvement_10_extra_col.csv",
    "submission_improvement_01_pseudo_label.csv",
    "submission_improvement_08_tfidf_svd.csv",
    "submission_improvement_03_stacking.csv",
    "submission_blend_weighted_3.csv",
    "submission_bpr64_seed_avg.csv",
    "submission_modernbert.csv",
    "submission_modernbert_time.csv",
    "submission_deberta_v3_base.csv",
    "submission_modernbert_nofill.csv",
    "submission_blend_best_bert.csv",
    "submission_blend_best_bert_07.csv",
    "submission_blend_best_bert_03.csv",
    "submission_blend_best_deberta.csv",
    "submission_blend_weighted_4.csv",
    "submission_blend_best_bert_06.csv",
]
for name in expected:
    p = ctx.submissions_dir / name
    print(f"  {'✓' if p.exists() else '✗'} {name}")
print(f"  合計: {sum(1 for n in expected if (ctx.submissions_dir / n).exists())} / {len(expected)} 本")